In [6]:
from datasets import load_dataset
import pandas as pd
from data import make_dataset
from model import build_tcn
from pipeline import TCNPipeline

2026-04-21 14:40:26.339211: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-21 14:40:26.380438: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-21 14:40:26.381085: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-21 14:40:27.790359: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [5]:
# Exemple : dataset tourisme (fictive local)

dataset = load_dataset("zaai-ai/time_series_dataset_residuals")
# Choisir le split 'train' (ou 'test')
train_split = dataset['train']

# Convertir ce split en DataFrame
df = train_split.to_pandas()

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df = df.set_index('Date')
df = df.groupby("Date").mean(numeric_only=True) # on aggrege les doublons

# Target
y = df['residuals'][:1000]

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

n_total = len(y)
# print(n_total) # DEBUG

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# Splits
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

# on normalise après le split pour éviter le leakage
mean_train = train.mean()
std_train = train.std()

train = (train - mean_train) / std_train
val   = (val - mean_train) / std_train
test  = (test - mean_train) / std_train

def regularize(series):
    return series.reindex(
        pd.date_range(series.index.min(), series.index.max(), freq="MS")
    ).interpolate()

print(df.index.duplicated().sum())

train = regularize(train)
val = regularize(val)
test = regularize(test)

train_val = pd.concat([train, val]).sort_index()

# # supprimer les doublons (si présents)
# # Vérifie les doublons
# print(train.index.duplicated().sum())
# print(val.index.duplicated().sum())
# print(train_val.index.duplicated().sum())
# print(test.index.duplicated().sum())
# train = train[~train.index.duplicated(keep='first')]
# val = val[~val.index.duplicated(keep='first')]
# train_val = train_val[~train_val.index.duplicated(keep='first')]
# test = test[~test.index.duplicated(keep='first')]

# print(train.index.is_monotonic_increasing)
# print(val.index.is_monotonic_increasing)
# print(train_val.index.is_monotonic_increasing)
# print(test.index.is_monotonic_increasing)
# # on trie par date et on réindexe en attribuant une fréquence

print(train.isna().sum())
print(val.isna().sum())
print(train_val.isna().sum())
print(test.isna().sum())

0
0
0
0
0


In [7]:
# Covariables dynamiques
X = df[['CPI', 'Inflation_Rate', 'GDP']][:1000]

# On standardise
X = (X - X.mean()) / X.std()

X_full = pd.concat([y, X], axis=1)

# Train
X_train = X_full.iloc[:train_end]

# Validation
X_val = X_full.iloc[train_end:val_end]

# Test
X_test = X.iloc[val_end:].astype(float)

# Train-Val
X_train_val = pd.concat([X_train, X_val])

# on standardise y
y = (y - y.mean()) / y.std()

data = pd.concat([y, X], axis=1)

In [8]:
y = pd.Series(y).reset_index(drop=True)
X = pd.DataFrame(X).reset_index(drop=True)

data = pd.concat([y, X], axis=1)
data.columns = ["target"] + list(X.columns)

In [9]:
# DATA
X_past, X_future, y = make_dataset(data, seq_len=30, horizon=10)

# MODEL
model = build_tcn(
    seq_len=30,
    n_past_features=3,
    horizon=10,
    n_future_features=3
)

# PIPELINE
pipe = TCNPipeline(model, seq_len=30, strategy="direct")

pipe.fit(X_past, X_future, y)

preds = pipe.predict(X_past[-1:], X_future[-1:])

Epoch 1/50
5/5 [==============================] - 1s 6ms/step - loss: 0.9601
Epoch 2/50
5/5 [==============================] - 0s 7ms/step - loss: 0.8429
Epoch 3/50
5/5 [==============================] - 0s 8ms/step - loss: 0.8014
Epoch 4/50
5/5 [==============================] - 0s 7ms/step - loss: 0.7723
Epoch 5/50
5/5 [==============================] - 0s 8ms/step - loss: 0.7511
Epoch 6/50
5/5 [==============================] - 0s 8ms/step - loss: 0.7440
Epoch 7/50
5/5 [==============================] - 0s 7ms/step - loss: 0.7308
Epoch 8/50
5/5 [==============================] - 0s 7ms/step - loss: 0.7215
Epoch 9/50
5/5 [==============================] - 0s 8ms/step - loss: 0.7102
Epoch 10/50
5/5 [==============================] - 0s 7ms/step - loss: 0.7025
Epoch 11/50
5/5 [==============================] - 0s 7ms/step - loss: 0.6976
Epoch 12/50
5/5 [==============================] - 0s 6ms/step - loss: 0.6926
Epoch 13/50
5/5 [==============================] - 0s 7ms/step - loss: 0.

In [11]:
preds

array([[ 0.53432155,  0.3123805 ,  0.3623311 ,  0.41786006, -0.16908982,
         0.55556196,  0.11936822, -0.13361895,  0.7235347 ,  0.93010795]],
      dtype=float32)